<a href="https://colab.research.google.com/github/shravya1998/indiaaihackathon_cdsco/blob/main/CDSCO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# For `transformers` backend
!pip install "mineru-vl-utils[transformers]"


In [ ]:
!pip install "mineru-vl-utils[vllm]"

In [ ]:
!pip install PyMuPDF

In [ ]:
!pip install transformers

In [ ]:
!pip install sentence-transformers faiss-cpu spacy pypdf

In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
# mineru_batch_extract.py

import os
import json
import fitz  # PyMuPDF
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed

import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from mineru_vl_utils import MinerUClient


MODEL_ID = "opendatalab/MinerU2.5-2509-1.2B"


# =========================
# 1. LOAD MODEL ONCE
# =========================
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    use_fast=True
)

client = MinerUClient(
    backend="transformers",
    model=model,
    processor=processor
)


# =========================
# 2. PDF -> PAGE IMAGES
# =========================
def pdf_to_images(pdf_path, dpi=200):
    doc = fitz.open(pdf_path)
    images = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        pix = page.get_pixmap(dpi=dpi)

        img = Image.frombytes(
            "RGB",
            [pix.width, pix.height],
            pix.samples
        )

        images.append((page_num + 1, img))

    return images


# =========================
# 3. CONVERT BLOCKS TO JSON
# =========================
def block_to_dict(block):
    return {
        "type": getattr(block, "type", None),
        "bbox": getattr(block, "bbox", None),
        "angle": getattr(block, "angle", None),
        "content": getattr(block, "content", None),
    }


# =========================
# 4. PROCESS ONE PDF
# =========================
def process_pdf(pdf_path, output_dir):
    pdf_path = Path(pdf_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    result = {
        "file_name": pdf_path.name,
        "pages": []
    }

    pages = pdf_to_images(pdf_path)

    for page_number, image in pages:
        print(f"Processing {pdf_path.name} | Page {page_number}")

        blocks = client.two_step_extract(image)

        page_result = {
            "page_number": page_number,
            "blocks": [block_to_dict(b) for b in blocks]
        }

        result["pages"].append(page_result)

    output_file = output_dir / f"{pdf_path.stem}.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    return str(output_file)


# =========================
# 5. MULTI-PDF THREADING
# =========================
def process_folder(input_folder, output_folder, max_workers=2):
    input_folder = Path(input_folder)
    pdf_files = list(input_folder.glob("*.pdf"))

    if not pdf_files:
        raise ValueError("No PDF files found in input folder.")

    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(process_pdf, pdf, output_folder): pdf
            for pdf in pdf_files
        }

        for future in as_completed(futures):
            pdf = futures[future]

            try:
                output_path = future.result()
                print(f"Completed: {pdf.name}")
                results.append(output_path)

            except Exception as e:
                print(f"Failed: {pdf.name} | Error: {e}")

    return results



In [3]:
!mkdir guidelines

In [ ]:
# ============================================================
# AI REGULATORY WORKFLOW:
# MinerU JSON → Anonymization → TF-IDF RAG → Compliance Report
# ============================================================

# Install in Colab:
# !pip install -q scikit-learn pypdf spacy
# !python -m spacy download en_core_web_sm

import os
import re
import json
import pickle
import hashlib
from pathlib import Path
from typing import Dict, Any, List, Tuple

from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# CONFIG
# ============================================================

MINERU_JSON_INPUT_FOLDER = "/content/mineru_outputs"
GUIDELINES_FOLDER = "/content/guidelines"

OUTPUT_FOLDER = "/content/final_outputs"
ANONYMIZED_FOLDER = f"{OUTPUT_FOLDER}/anonymized"
REPORT_FOLDER = f"{OUTPUT_FOLDER}/reports"
RAG_INDEX_FOLDER = f"{OUTPUT_FOLDER}/rag_index"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 5

USE_NLP = True

ANONYMIZATION_MODE = "pseudonymize"
# Options:
# "pseudonymize"  -> secure reversible tokens using entity map
# "irreversible" -> generalized non-reversible tokens


# ============================================================
# 1. MINERU JSON TO TEXT
# ============================================================

def mineru_json_to_text(json_path: str) -> str:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    lines = []

    for page in data.get("pages", []):
        page_no = page.get("page_number", "")
        lines.append(f"\n--- Page {page_no} ---")

        for block in page.get("blocks", []):
            block_type = block.get("type", "unknown")
            content = block.get("content", "")
            lines.append(f"[{block_type}] {content}")

    return "\n".join(lines)


# ============================================================
# 2. HYBRID ANONYMIZER
# ============================================================

class HybridAnonymizer:
    def __init__(self, mode: str = "pseudonymize"):
        self.mode = mode
        self.entity_map = {}
        self.counters = {
            "PERSON": 0,
            "ORG": 0,
            "ADDRESS": 0,
            "EMAIL": 0,
            "PHONE": 0,
            "PINCODE": 0,
            "PATIENT_ID": 0,
            "TRIAL_ID": 0,
            "HOSPITAL": 0,
            "DATE": 0,
            "AGE": 0,
            "PHI": 0,
            "ID": 0
        }

    def token(self, value: str, entity_type: str) -> str:
        value = str(value).strip()

        if not value:
            return ""

        if value in self.entity_map:
            return self.entity_map[value]["token"]

        self.counters[entity_type] += 1

        if self.mode == "pseudonymize":
            digest = hashlib.sha256(value.encode("utf-8")).hexdigest()[:8]
            token = f"[{entity_type}_{self.counters[entity_type]}_{digest}]"
        else:
            token = self.generalize(value, entity_type)

        self.entity_map[value] = {
            "entity_type": entity_type,
            "token": token
        }

        return token

    def generalize(self, value: str, entity_type: str) -> str:
        if entity_type == "AGE":
            nums = re.findall(r"\d+", value)
            if nums:
                age = int(nums[0])
                if age < 18:
                    return "[AGE_GROUP_CHILD]"
                elif age <= 40:
                    return "[AGE_GROUP_18_40]"
                elif age <= 60:
                    return "[AGE_GROUP_41_60]"
                else:
                    return "[AGE_GROUP_60_PLUS]"
            return "[AGE_GENERALIZED]"

        if entity_type == "DATE":
            return "[DATE_GENERALIZED]"

        if entity_type == "ADDRESS":
            return "[LOCATION_GENERALIZED]"

        return f"[{entity_type}_REMOVED]"


# ============================================================
# 3. RULE-BASED PII / PHI ANONYMIZATION
# ============================================================

def rule_based_anonymize(text: str, anonymizer: HybridAnonymizer) -> str:

    field_patterns = [
        ("ORG", r"(Company|Sponsor|Manufacturer|Applicant|Organization):\s*(.+)"),
        ("ADDRESS", r"(Address|Site Address|Residential Address):\s*(.+)"),
        ("PERSON", r"(Authorized Person|Principal Investigator|Investigator|Patient Name|Subject Name|Doctor Name):\s*(.+)"),
        ("HOSPITAL", r"(Hospital|Clinical Site|Trial Site):\s*(.+)"),
        ("PATIENT_ID", r"(Patient ID|Subject ID|Participant ID):\s*(.+)"),
        ("TRIAL_ID", r"(Trial ID|Protocol No|Protocol Number):\s*(.+)"),
        ("AGE", r"(Age):\s*(\d{1,3})"),
    ]

    for entity_type, pattern in field_patterns:
        def replace_field(match, et=entity_type):
            label = match.group(1)
            value = match.group(2)
            return f"{label}: {anonymizer.token(value, et)}"

        text = re.sub(pattern, replace_field, text, flags=re.IGNORECASE)

    regex_patterns = {
        "EMAIL": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        "PHONE": r"\b(?:\+91[-\s]?)?[6-9]\d{9}\b",
        "PINCODE": r"\b\d{6}\b",
        "DATE": r"\b(?:\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\d{4}-\d{2}-\d{2})\b",
        "PATIENT_ID": r"\b(?:PAT|PID|PT|SUBJ|SUBJECT)[-_]?\d{3,10}\b",
        "ID": r"\b(?:AADHAAR|PAN|PASSPORT|DL)[-:\s]?[A-Z0-9]{6,20}\b"
    }

    for entity_type, pattern in regex_patterns.items():
        text = re.sub(
            pattern,
            lambda m, et=entity_type: anonymizer.token(m.group(0), et),
            text,
            flags=re.IGNORECASE
        )

    return text


# ============================================================
# 4. NLP-BASED ANONYMIZATION
# ============================================================

def nlp_anonymize(text: str, anonymizer: HybridAnonymizer) -> str:
    try:
        import spacy
        nlp = spacy.load("en_core_web_sm")
    except Exception:
        print("spaCy not available. Skipping NLP.")
        return text

    doc = nlp(text)
    replacements = []

    for ent in doc.ents:
        if ent.label_ == "PERSON":
            entity_type = "PERSON"
        elif ent.label_ == "ORG":
            entity_type = "ORG"
        elif ent.label_ in ["GPE", "LOC", "FAC"]:
            entity_type = "ADDRESS"
        elif ent.label_ == "DATE":
            entity_type = "DATE"
        else:
            continue

        token = anonymizer.token(ent.text, entity_type)
        replacements.append((ent.start_char, ent.end_char, token))

    for start, end, token in sorted(replacements, reverse=True):
        text = text[:start] + token + text[end:]

    return text


def anonymize_document(text: str, mode: str = "pseudonymize", use_nlp: bool = True):
    anonymizer = HybridAnonymizer(mode=mode)

    anonymized = rule_based_anonymize(text, anonymizer)

    if use_nlp:
        anonymized = nlp_anonymize(anonymized, anonymizer)

    risk_report = residual_pii_check(anonymized)

    return anonymized, anonymizer.entity_map, risk_report


# ============================================================
# 5. RESIDUAL PII / PHI RISK CHECK
# ============================================================

def residual_pii_check(text: str) -> Dict[str, Any]:
    checks = {
        "email": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        "phone": r"\b(?:\+91[-\s]?)?[6-9]\d{9}\b",
        "pincode": r"\b\d{6}\b",
        "date": r"\b(?:\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\d{4}-\d{2}-\d{2})\b",
        "patient_id": r"\b(?:PAT|PID|PT|SUBJ|SUBJECT)[-_]?\d{3,10}\b"
    }

    findings = []

    for risk_type, pattern in checks.items():
        matches = list(set(re.findall(pattern, text, flags=re.IGNORECASE)))
        if matches:
            findings.append({
                "risk_type": risk_type,
                "matches": matches[:10]
            })

    return {
        "status": "PASS" if not findings else "REVIEW_REQUIRED",
        "findings": findings
    }


# ============================================================
# 6. GUIDELINE LOADING
# ============================================================

def read_pdf_text(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    text = []

    for page in reader.pages:
        try:
            text.append(page.extract_text() or "")
        except Exception:
            text.append("")

    return "\n".join(text)


def load_guideline_documents(folder: str) -> List[Dict[str, str]]:
    docs = []
    folder_path = Path(folder)

    for file in folder_path.glob("*"):
        suffix = file.suffix.lower()

        if suffix == ".pdf":
            text = read_pdf_text(str(file))
        elif suffix in [".txt", ".md"]:
            text = file.read_text(encoding="utf-8", errors="ignore")
        else:
            continue

        if text.strip():
            docs.append({
                "source": file.name,
                "text": text
            })

    return docs


def chunk_text(text: str, source: str) -> List[Dict[str, Any]]:
    chunks = []
    start = 0
    chunk_id = 0

    while start < len(text):
        end = start + CHUNK_SIZE
        chunk = text[start:end]

        if chunk.strip():
            chunks.append({
                "source": source,
                "chunk_id": chunk_id,
                "text": chunk
            })

        chunk_id += 1
        start += CHUNK_SIZE - CHUNK_OVERLAP

    return chunks


# ============================================================
# 7. TF-IDF RAG INDEX
# ============================================================

def build_rag_index(guidelines_folder: str, index_folder: str):
    Path(index_folder).mkdir(parents=True, exist_ok=True)

    docs = load_guideline_documents(guidelines_folder)

    if not docs:
        raise ValueError(
            f"No guideline files found in {guidelines_folder}. "
            "Add DPDP, NDHM/ABDM, ICMR, CDSCO PDFs/TXT files."
        )

    chunks = []

    for doc in docs:
        chunks.extend(chunk_text(doc["text"], doc["source"]))

    texts = [chunk["text"] for chunk in chunks]

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=30000,
        ngram_range=(1, 2)
    )

    matrix = vectorizer.fit_transform(texts)

    with open(Path(index_folder) / "vectorizer.pkl", "wb") as f:
        pickle.dump(vectorizer, f)

    with open(Path(index_folder) / "matrix.pkl", "wb") as f:
        pickle.dump(matrix, f)

    with open(Path(index_folder) / "chunks.json", "w", encoding="utf-8") as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)

    print(f"RAG index created with {len(chunks)} chunks.")


def load_rag_index(index_folder: str):
    with open(Path(index_folder) / "vectorizer.pkl", "rb") as f:
        vectorizer = pickle.load(f)

    with open(Path(index_folder) / "matrix.pkl", "rb") as f:
        matrix = pickle.load(f)

    with open(Path(index_folder) / "chunks.json", "r", encoding="utf-8") as f:
        chunks = json.load(f)

    return vectorizer, matrix, chunks


def retrieve_guidelines(
    query: str,
    vectorizer,
    matrix,
    chunks,
    top_k: int = TOP_K
) -> List[Dict[str, Any]]:

    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, matrix)[0]

    top_indices = scores.argsort()[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append({
            "score": float(scores[idx]),
            "source": chunks[idx]["source"],
            "chunk_id": chunks[idx]["chunk_id"],
            "text": chunks[idx]["text"]
        })

    return results


# ============================================================
# 8. COMPLIANCE CHECK
# ============================================================

def check_compliance(
    anonymized_text: str,
    risk_report: Dict[str, Any],
    vectorizer,
    matrix,
    chunks
) -> Dict[str, Any]:

    retrieval_queries = {
        "DPDP privacy and anonymisation": (
            "personal data data fiduciary consent purpose limitation "
            "data minimisation anonymisation pseudonymisation security safeguards"
        ),
        "NDHM ABDM health data privacy": (
            "health data consent privacy confidentiality anonymisation "
            "electronic health records data sharing"
        ),
        "ICMR ethics clinical research": (
            "clinical trial participant confidentiality informed consent "
            "ethics committee privacy risk benefit"
        ),
        "CDSCO regulatory review": (
            "new drug application clinical trial safety efficacy toxicology "
            "pharmacology chemistry pharmaceutical data"
        )
    }

    retrieved_context = []

    for category, query in retrieval_queries.items():
        results = retrieve_guidelines(query, vectorizer, matrix, chunks)
        for r in results:
            r["category"] = category
            retrieved_context.append(r)

    unique = {}
    for item in retrieved_context:
        key = f"{item['source']}::{item['chunk_id']}"
        unique[key] = item

    retrieved_context = list(unique.values())

    controls = []

    # Control 1: residual PII/PHI
    controls.append({
        "control": "Residual PII/PHI detection",
        "status": "PASS" if risk_report["status"] == "PASS" else "REVIEW_REQUIRED",
        "evidence": (
            "No residual direct identifiers detected."
            if risk_report["status"] == "PASS"
            else f"Residual identifiers found: {risk_report['findings']}"
        )
    })

    # Control 2: tokenization applied
    token_pattern = r"\[(PERSON|ORG|ADDRESS|EMAIL|PHONE|PINCODE|PATIENT_ID|TRIAL_ID|HOSPITAL|DATE|AGE|PHI|ID)"
    has_tokens = bool(re.search(token_pattern, anonymized_text))

    controls.append({
        "control": "Pseudonymisation / irreversible anonymisation",
        "status": "PASS" if has_tokens else "REVIEW_REQUIRED",
        "evidence": (
            "Sensitive identifiers were replaced with secure/generalized tokens."
            if has_tokens
            else "No anonymization tokens found."
        )
    })

    # Control 3: CDSCO document completeness
    required_sections = [
        "Applicant Details",
        "Drug Details",
        "Composition",
        "Chemical",
        "Pharmaceutical",
        "Animal Pharmacology",
        "Toxicology",
        "Clinical Trials",
        "Regulatory Status",
        "Prescribing Info"
    ]

    missing_sections = [
        s for s in required_sections
        if s.lower() not in anonymized_text.lower()
    ]

    controls.append({
        "control": "CDSCO submission completeness",
        "status": "PASS" if not missing_sections else "REVIEW_REQUIRED",
        "evidence": (
            "Required regulatory sections are present."
            if not missing_sections
            else f"Missing sections: {missing_sections}"
        )
    })

    # Control 4: privacy-by-design workflow
    workflow_pass = has_tokens and risk_report["status"] == "PASS"

    controls.append({
        "control": "Privacy-by-design processing",
        "status": "PASS" if workflow_pass else "REVIEW_REQUIRED",
        "evidence": (
            "Document passed anonymization and residual-risk checks before compliance review."
            if workflow_pass
            else "Document needs reviewer validation before regulatory use."
        )
    })

    overall_status = (
        "PASS"
        if all(c["status"] == "PASS" for c in controls)
        else "REVIEW_REQUIRED"
    )

    return {
        "overall_status": overall_status,
        "controls": controls,
        "retrieved_guideline_context": retrieved_context[:10]
    }


# ============================================================
# 9. PROCESS ALL MINERU JSON FILES
# ============================================================

def process_all_documents():
    Path(ANONYMIZED_FOLDER).mkdir(parents=True, exist_ok=True)
    Path(REPORT_FOLDER).mkdir(parents=True, exist_ok=True)

    if not Path(RAG_INDEX_FOLDER, "vectorizer.pkl").exists():
        build_rag_index(GUIDELINES_FOLDER, RAG_INDEX_FOLDER)

    vectorizer, matrix, chunks = load_rag_index(RAG_INDEX_FOLDER)

    json_files = list(Path(MINERU_JSON_INPUT_FOLDER).glob("*.json"))

    print(f"Found {len(json_files)} MinerU JSON files.")

    for json_file in json_files:
        print(f"\nProcessing: {json_file.name}")

        original_text = mineru_json_to_text(str(json_file))

        anonymized_text, entity_map, risk_report = anonymize_document(
            original_text,
            mode=ANONYMIZATION_MODE,
            use_nlp=USE_NLP
        )

        compliance_report = check_compliance(
            anonymized_text=anonymized_text,
            risk_report=risk_report,
            vectorizer=vectorizer,
            matrix=matrix,
            chunks=chunks
        )

        base = json_file.stem

        with open(Path(ANONYMIZED_FOLDER) / f"{base}_anonymized.txt", "w", encoding="utf-8") as f:
            f.write(anonymized_text)

        with open(Path(ANONYMIZED_FOLDER) / f"{base}_entity_map.json", "w", encoding="utf-8") as f:
            json.dump(entity_map, f, indent=2, ensure_ascii=False)

        with open(Path(REPORT_FOLDER) / f"{base}_risk_report.json", "w", encoding="utf-8") as f:
            json.dump(risk_report, f, indent=2, ensure_ascii=False)

        with open(Path(REPORT_FOLDER) / f"{base}_compliance_report.json", "w", encoding="utf-8") as f:
            json.dump(compliance_report, f, indent=2, ensure_ascii=False)

        print("Saved:")
        print(f"- {Path(ANONYMIZED_FOLDER) / f'{base}_anonymized.txt'}")
        print(f"- {Path(REPORT_FOLDER) / f'{base}_compliance_report.json'}")


if __name__ == "__main__":
    process_all_documents()

In [ ]:
# eval_anonymized_folder.py

import re
import json
from pathlib import Path
from collections import Counter, defaultdict

ANONYMIZED_FOLDER = "/content/final_outputs/anonymized"
OUTPUT_JSON = "/content/final_outputs/anonymized_eval_metrics.json"

QUASI_IDENTIFIER_TYPES = ["AGE", "DATE", "PINCODE", "ADDRESS"]
SENSITIVE_TYPES = ["PATIENT_ID", "TRIAL_ID", "ID", "PHI"]


def extract_tokens(text):
    return re.findall(r"\[([A-Z_]+)(?:_\d+)?(?:_[a-f0-9]{8})?\]", text)


def residual_pii_check(text):
    patterns = {
        "email": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        "phone": r"\b(?:\+91[-\s]?)?[6-9]\d{9}\b",
        "pincode": r"\b\d{6}\b",
        "date": r"\b(?:\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\d{4}-\d{2}-\d{2})\b",
        "patient_id": r"\b(?:PAT|PID|PT|SUBJ|SUBJECT)[-_]?\d{3,10}\b",
    }

    findings = {}

    for key, pattern in patterns.items():
        matches = list(set(re.findall(pattern, text, flags=re.IGNORECASE)))
        if matches:
            findings[key] = matches[:10]

    return findings


def build_records_from_files(folder):
    records = []

    for file in Path(folder).glob("*.txt"):
        text = file.read_text(encoding="utf-8", errors="ignore")
        tokens = extract_tokens(text)
        token_counts = Counter(tokens)

        residual_findings = residual_pii_check(text)

        record = {
            "file_name": file.name,
            "text_length": len(text),
            "token_counts": dict(token_counts),
            "residual_findings": residual_findings,
        }

        for qi in QUASI_IDENTIFIER_TYPES:
            record[qi] = token_counts.get(qi, 0)

        sensitive_value = "NONE"
        for s in SENSITIVE_TYPES:
            if token_counts.get(s, 0) > 0:
                sensitive_value = s
                break

        record["sensitive_attribute"] = sensitive_value
        records.append(record)

    return records


def calculate_k_anonymity(records):
    groups = Counter()

    for r in records:
        key = tuple(r.get(qi, 0) for qi in QUASI_IDENTIFIER_TYPES)
        groups[key] += 1

    if not groups:
        return 0, {}

    return min(groups.values()), {str(k): v for k, v in groups.items()}


def calculate_l_diversity(records):
    groups = defaultdict(set)

    for r in records:
        key = tuple(r.get(qi, 0) for qi in QUASI_IDENTIFIER_TYPES)
        groups[key].add(r["sensitive_attribute"])

    if not groups:
        return 0, {}

    l_values = {str(k): len(v) for k, v in groups.items()}
    return min(l_values.values()), l_values


def distribution(values):
    total = len(values)
    counts = Counter(values)

    if total == 0:
        return {}

    return {k: v / total for k, v in counts.items()}


def tv_distance(d1, d2):
    keys = set(d1) | set(d2)
    return 0.5 * sum(abs(d1.get(k, 0) - d2.get(k, 0)) for k in keys)


def calculate_t_closeness(records):
    overall = distribution([r["sensitive_attribute"] for r in records])

    groups = defaultdict(list)
    for r in records:
        key = tuple(r.get(qi, 0) for qi in QUASI_IDENTIFIER_TYPES)
        groups[key].append(r["sensitive_attribute"])

    scores = {}

    for key, values in groups.items():
        group_dist = distribution(values)
        scores[str(key)] = tv_distance(overall, group_dist)

    max_t = max(scores.values()) if scores else 0.0
    return max_t, scores


def evaluate_folder(folder):
    records = build_records_from_files(folder)

    total_files = len(records)
    files_with_residual_pii = sum(
        1 for r in records if r["residual_findings"]
    )

    total_tokens = Counter()
    for r in records:
        total_tokens.update(r["token_counts"])

    k, groups = calculate_k_anonymity(records)
    l, l_values = calculate_l_diversity(records)
    t, t_scores = calculate_t_closeness(records)

    report = {
        "total_files": total_files,
        "files_with_residual_pii": files_with_residual_pii,
        "residual_pii_rate": files_with_residual_pii / total_files if total_files else 0,
        "total_anonymization_tokens": dict(total_tokens),
        "k_anonymity": k,
        "l_diversity": l,
        "t_closeness": t,
        "status": {
            "residual_pii": "PASS" if files_with_residual_pii == 0 else "REVIEW_REQUIRED",
            "k_anonymity": "PASS" if k >= 5 else "REVIEW_REQUIRED",
            "l_diversity": "PASS" if l >= 2 else "REVIEW_REQUIRED",
            "t_closeness": "PASS" if t <= 0.2 else "REVIEW_REQUIRED",
        },
        "file_level_records": records,
        "equivalence_classes": groups,
        "l_values": l_values,
        "t_scores": t_scores,
    }

    return report


def print_report(report):
    print("\n📊 ANONYMIZED FILE EVALUATION METRICS")
    print("=" * 60)

    print(f"Total files evaluated: {report['total_files']}")
    print(f"Files with residual PII: {report['files_with_residual_pii']}")
    print(f"Residual PII Rate: {report['residual_pii_rate']:.4f}")

    print("\n🔐 Privacy Metrics")
    print(f"k-Anonymity: {report['k_anonymity']}")
    print(f"l-Diversity: {report['l_diversity']}")
    print(f"t-Closeness: {report['t_closeness']:.4f}")

    print("\n🏷️ Anonymization Token Counts")
    for token, count in report["total_anonymization_tokens"].items():
        print(f"{token}: {count}")

    print("\n✅ Status")
    for metric, status in report["status"].items():
        print(f"{metric}: {status}")

    print("=" * 60)


if __name__ == "__main__":
    report = evaluate_folder(ANONYMIZED_FOLDER)

    Path(OUTPUT_JSON).parent.mkdir(parents=True, exist_ok=True)

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    print_report(report)

    print(f"\nSaved evaluation report to: {OUTPUT_JSON}")

In [ ]:
# =========================
# MedGemma Regulatory Summarizer (Single File - No argparse)
# =========================

import os
import json
from pathlib import Path
from typing import List, Dict

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


# =========================
# CONFIG
# =========================

MODEL_NAME = "google/gemma-3-1b-it"

SUPPORTED_EXTENSIONS = {".txt", ".md", ".json"}


# =========================
# USER INPUT (EDIT HERE)
# =========================

INPUT_FOLDER = "/content/final_outputs/anonymized"
OUTPUT_FOLDER = "/content/regulatory_summaries"

# Choose: "sugam", "sae", "meeting"
DOCUMENT_TYPE = "sae"


# =========================
# OUTPUT SCHEMAS
# =========================

OUTPUT_SCHEMAS = {
    "sugam": """
DOCUMENT TYPE: SUGAM APPLICATION CHECKLIST SUMMARY

APPLICATION OVERVIEW:
- Application ID:
- Product/Drug/Device Name:
- Applicant/Sponsor:
- Submission Type:
- Review Category:

CHECKLIST COMPLETENESS:
- Total checklist items:
- Completed items:
- Missing items:
- Incomplete/unclear items:

CRITICAL REGULATORY FINDINGS:
1.
2.
3.

MISSING / DEFICIENT DOCUMENTS:
1.
2.
3.

REVIEWER ACTION REQUIRED:
- Accept for review / Query applicant / Reject as incomplete

SUMMARY FOR REVIEWER:
""",

    "sae": """
DOCUMENT TYPE: SAE CASE SUMMARY

CASE OVERVIEW:
- Case ID:
- Patient ID:
- Trial ID:
- Suspect Drug/Intervention:
- Event Term:
- Event Date:
- Seriousness Criteria:

CLINICAL NARRATIVE SUMMARY:

CAUSALITY INDICATORS:
- Temporal relationship:
- Dechallenge/Rechallenge:
- Alternative causes:
- Concomitant medication:
- Investigator causality:
- Sponsor causality:

REGULATORY ASSESSMENT:
- Expectedness:
- Severity:
- Outcome:
- Reportability:

KEY GAPS / QUERIES:
1.
2.
3.

REVIEWER DECISION SUPPORT:
""",

    "meeting": """
DOCUMENT TYPE: REGULATORY MEETING SUMMARY

MEETING OVERVIEW:
- Meeting Date:
- Participants:
- Agenda:
- Submission/Product Discussed:

KEY DISCUSSIONS:
1.
2.
3.

DECISIONS TAKEN:
1.
2.
3.

ACTION ITEMS:
- Action:
  Owner:
  Due Date:
  Priority:

RISKS / OPEN ISSUES:
1.
2.
3.

NEXT STEPS:
1.
2.
3.

REVIEWER SUMMARY:
"""
}


# =========================
# LOAD MODEL
# =========================

def load_model():
    print("Loading MedGemma...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    return tokenizer, model


# =========================
# FILE READER
# =========================

def read_file(file_path: Path) -> str:
    suffix = file_path.suffix.lower()

    if suffix in {".txt", ".md"}:
        return file_path.read_text(encoding="utf-8", errors="ignore")

    if suffix == ".json":
        data = json.loads(file_path.read_text(encoding="utf-8", errors="ignore"))
        return json.dumps(data, indent=2, ensure_ascii=False)

    return ""


# =========================
# TEXT CHUNKING
# =========================

def chunk_text(text: str, max_chars=6000, overlap=500):
    chunks = []
    start = 0

    while start < len(text):
        end = start + max_chars
        chunks.append(text[start:end])
        start = end - overlap

    return chunks


# =========================
# PROMPTS
# =========================

def build_chunk_prompt(doc_type, text):
    return f"""
You are a medical regulatory reviewer assistant.

Extract only critical regulatory information.

Rules:
- No hallucination
- If missing → "Not specified"
- Focus on regulatory, safety, checklist, decisions

DOCUMENT TYPE: {doc_type}

TEXT:
{text}

Return concise summary.
"""


def build_final_prompt(doc_type, combined):
    return f"""
Convert extracted info into this format:

{OUTPUT_SCHEMAS[doc_type]}

Rules:
- No hallucination
- Missing → "Not specified"
- Concise
- Highlight risks, missing data, actions

DATA:
{combined}
"""


# =========================
# GENERATION
# =========================

def generate(prompt, tokenizer, model, max_tokens=1200):
    messages = [{"role": "user", "content": prompt}]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_tokens,
            temperature=0.2,
            do_sample=False
        )

    response = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip()


# =========================
# PROCESS SINGLE FILE
# =========================

def process_file(file_path, tokenizer, model):
    print(f"\nProcessing: {file_path.name}")

    text = read_file(file_path)
    if not text:
        return None

    chunks = chunk_text(text)

    summaries = []
    for i, chunk in enumerate(chunks):
        print(f"Chunk {i+1}/{len(chunks)}")

        prompt = build_chunk_prompt(DOCUMENT_TYPE, chunk)
        summary = generate(prompt, tokenizer, model, 800)
        summaries.append(summary)

    combined = "\n".join(summaries)

    final_prompt = build_final_prompt(DOCUMENT_TYPE, combined)
    final_summary = generate(final_prompt, tokenizer, model, 1500)

    return final_summary


# =========================
# MAIN PIPELINE
# =========================

def run_pipeline():
    input_path = Path(INPUT_FOLDER)
    output_path = Path(OUTPUT_FOLDER)

    output_path.mkdir(parents=True, exist_ok=True)

    tokenizer, model = load_model()

    files = [
        f for f in input_path.iterdir()
        if f.suffix.lower() in SUPPORTED_EXTENSIONS
    ]

    if not files:
        print("No files found")
        return

    for file in files:
        try:
            summary = process_file(file, tokenizer, model)

            if summary:
                out_file = output_path / f"{file.stem}_summary.txt"
                out_file.write_text(summary, encoding="utf-8")

        except Exception as e:
            print(f"Error in {file.name}: {e}")

    print("\nDone! Summaries saved.")


# =========================
# RUN
# =========================

run_pipeline()

In [ ]:
# completeness_and_comparison.py

import re
import json
import difflib
from pathlib import Path
from typing import Dict, Any, List


# ============================================================
# CONFIG
# ============================================================

CURRENT_FOLDER = "/content/final_outputs/anonymized"
PREVIOUS_FOLDER = "/content/previous_docs_text"
OUTPUT_FOLDER = "/content/completeness_outputs"

# Example mandatory schemas
APPLICATION_REQUIRED_FIELDS = [
    "Applicant Name",
    "Application Number",
    "Drug Name",
    "Dosage Form",
    "Strength",
    "Manufacturer",
    "Clinical Trial Protocol Number",
    "Principal Investigator",
    "Trial Site",
    "Ethics Committee Approval",
    "Informed Consent",
    "Study Objective",
    "Inclusion Criteria",
    "Exclusion Criteria",
    "Safety Data",
    "Efficacy Data",
    "Signature",
    "Date"
]

SAE_REQUIRED_FIELDS = [
    "Patient ID",
    "Age",
    "Gender",
    "SAE Report Number",
    "Event Term",
    "Date of Onset",
    "Date of Report",
    "Suspected Drug",
    "Dose",
    "Route",
    "Causality Assessment",
    "Outcome",
    "Seriousness Criteria",
    "Reporter Name",
    "Investigator Signature"
]

CONSISTENCY_FIELDS = [
    "Applicant Name",
    "Application Number",
    "Drug Name",
    "Clinical Trial Protocol Number",
    "Patient ID",
    "SAE Report Number",
    "Suspected Drug"
]


# ============================================================
# BASIC TEXT LOADING
# ============================================================

def read_text_file(path: str) -> str:
    return Path(path).read_text(encoding="utf-8", errors="ignore")


def load_folder_texts(folder: str) -> Dict[str, str]:
    docs = {}

    for file in Path(folder).glob("*.txt"):
        docs[file.name] = read_text_file(str(file))

    return docs


# ============================================================
# FIELD EXTRACTION
# ============================================================

def extract_field_value(text: str, field_name: str) -> str:
    """
    Extracts values like:
    Applicant Name: ABC Pharma
    Applicant Name - ABC Pharma
    Applicant Name    ABC Pharma
    """

    pattern = rf"{re.escape(field_name)}\s*[:\-]\s*(.+)"
    match = re.search(pattern, text, flags=re.IGNORECASE)

    if match:
        return match.group(1).strip()

    return ""


def field_exists(text: str, field_name: str) -> bool:
    value = extract_field_value(text, field_name)

    if value:
        return True

    return bool(re.search(re.escape(field_name), text, flags=re.IGNORECASE))


def extract_all_fields(text: str, fields: List[str]) -> Dict[str, str]:
    extracted = {}

    for field in fields:
        extracted[field] = extract_field_value(text, field)

    return extracted


# ============================================================
# COMPLETENESS ASSESSMENT
# ============================================================

def assess_completeness(
    text: str,
    required_fields: List[str],
    document_type: str
) -> Dict[str, Any]:

    present = []
    missing = []

    for field in required_fields:
        if field_exists(text, field):
            present.append(field)
        else:
            missing.append(field)

    total = len(required_fields)
    score = len(present) / total if total else 0

    return {
        "document_type": document_type,
        "total_required_fields": total,
        "present_fields": present,
        "missing_fields": missing,
        "completeness_score": round(score, 4),
        "status": "PASS" if score >= 0.9 and not missing else "REVIEW_REQUIRED"
    }


# ============================================================
# CONSISTENCY CHECK
# ============================================================

def check_consistency_across_documents(
    documents: Dict[str, str],
    consistency_fields: List[str]
) -> Dict[str, Any]:

    field_values = {}

    for file_name, text in documents.items():
        for field in consistency_fields:
            value = extract_field_value(text, field)

            if value:
                field_values.setdefault(field, {})
                field_values[field][file_name] = value

    inconsistencies = []

    for field, values_by_file in field_values.items():
        unique_values = set(v.lower().strip() for v in values_by_file.values())

        if len(unique_values) > 1:
            inconsistencies.append({
                "field": field,
                "values_by_file": values_by_file
            })

    return {
        "checked_fields": consistency_fields,
        "inconsistencies": inconsistencies,
        "status": "PASS" if not inconsistencies else "REVIEW_REQUIRED"
    }


# ============================================================
# TEXT VERSION COMPARISON
# ============================================================

def compare_text_versions(
    old_text: str,
    new_text: str
) -> Dict[str, Any]:

    old_lines = old_text.splitlines()
    new_lines = new_text.splitlines()

    diff = list(difflib.unified_diff(
        old_lines,
        new_lines,
        fromfile="previous_version",
        tofile="current_version",
        lineterm=""
    ))

    added = []
    removed = []

    for line in diff:
        if line.startswith("+") and not line.startswith("+++"):
            added.append(line[1:])
        elif line.startswith("-") and not line.startswith("---"):
            removed.append(line[1:])

    similarity = difflib.SequenceMatcher(
        None,
        old_text,
        new_text
    ).ratio()

    return {
        "similarity_score": round(similarity, 4),
        "added_lines": added[:100],
        "removed_lines": removed[:100],
        "total_added_lines": len(added),
        "total_removed_lines": len(removed),
        "status": "CHANGES_FOUND" if added or removed else "NO_CHANGE"
    }


# ============================================================
# FIELD-LEVEL VERSION COMPARISON
# ============================================================

def compare_field_changes(
    old_text: str,
    new_text: str,
    fields: List[str]
) -> Dict[str, Any]:

    changes = []

    for field in fields:
        old_value = extract_field_value(old_text, field)
        new_value = extract_field_value(new_text, field)

        if old_value != new_value:
            changes.append({
                "field": field,
                "previous_value": old_value,
                "current_value": new_value,
                "change_type": classify_change(old_value, new_value)
            })

    return {
        "field_changes": changes,
        "total_field_changes": len(changes),
        "status": "CHANGES_FOUND" if changes else "NO_FIELD_CHANGE"
    }


def classify_change(old_value: str, new_value: str) -> str:
    if not old_value and new_value:
        return "ADDED"
    if old_value and not new_value:
        return "REMOVED"
    if old_value != new_value:
        return "MODIFIED"
    return "UNCHANGED"


# ============================================================
# TABLE-LIKE COMPARISON
# ============================================================

def extract_table_like_lines(text: str) -> List[str]:
    """
    Simple table detector for reconstructed text.
    Works with lines containing | or multiple spaces/tabs.
    """

    lines = []

    for line in text.splitlines():
        if "|" in line or "\t" in line or re.search(r"\s{3,}", line):
            cleaned = line.strip()
            if len(cleaned) > 5:
                lines.append(cleaned)

    return lines


def compare_table_like_content(
    old_text: str,
    new_text: str
) -> Dict[str, Any]:

    old_tables = extract_table_like_lines(old_text)
    new_tables = extract_table_like_lines(new_text)

    old_set = set(old_tables)
    new_set = set(new_tables)

    added_rows = list(new_set - old_set)
    removed_rows = list(old_set - new_set)

    return {
        "added_table_rows": added_rows[:100],
        "removed_table_rows": removed_rows[:100],
        "total_added_table_rows": len(added_rows),
        "total_removed_table_rows": len(removed_rows),
        "status": "TABLE_CHANGES_FOUND" if added_rows or removed_rows else "NO_TABLE_CHANGE"
    }


# ============================================================
# SIGNIFICANT CHANGE FLAGGING
# ============================================================

def flag_significant_changes(
    text_diff: Dict[str, Any],
    field_diff: Dict[str, Any],
    table_diff: Dict[str, Any]
) -> Dict[str, Any]:

    flags = []

    critical_keywords = [
        "death",
        "serious adverse event",
        "sae",
        "causality",
        "fatal",
        "withdrawn",
        "dose changed",
        "contraindication",
        "safety",
        "efficacy",
        "protocol amendment",
        "ethics committee"
    ]

    changed_text = " ".join(
        text_diff.get("added_lines", []) +
        text_diff.get("removed_lines", [])
    ).lower()

    for keyword in critical_keywords:
        if keyword in changed_text:
            flags.append({
                "risk_type": "CRITICAL_TEXT_CHANGE",
                "keyword": keyword,
                "message": f"Significant regulatory/safety keyword changed: {keyword}"
            })

    for change in field_diff.get("field_changes", []):
        if change["field"] in [
            "Drug Name",
            "Clinical Trial Protocol Number",
            "Patient ID",
            "SAE Report Number",
            "Causality Assessment",
            "Outcome"
        ]:
            flags.append({
                "risk_type": "CRITICAL_FIELD_CHANGE",
                "field": change["field"],
                "change_type": change["change_type"],
                "message": f"Critical field changed: {change['field']}"
            })

    if table_diff.get("total_added_table_rows", 0) > 0 or table_diff.get("total_removed_table_rows", 0) > 0:
        flags.append({
            "risk_type": "TABLE_CHANGE",
            "message": "Table-like data changed between versions."
        })

    return {
        "significant_change_flags": flags,
        "status": "REVIEW_REQUIRED" if flags else "PASS"
    }


# ============================================================
# MAIN DOCUMENT ASSESSMENT
# ============================================================

def assess_single_document(
    file_name: str,
    text: str,
    document_type: str = "APPLICATION"
) -> Dict[str, Any]:

    if document_type.upper() == "SAE":
        required_fields = SAE_REQUIRED_FIELDS
    else:
        required_fields = APPLICATION_REQUIRED_FIELDS

    completeness = assess_completeness(
        text=text,
        required_fields=required_fields,
        document_type=document_type
    )

    extracted_fields = extract_all_fields(
        text=text,
        fields=required_fields
    )

    return {
        "file_name": file_name,
        "document_type": document_type,
        "completeness": completeness,
        "extracted_fields": extracted_fields
    }


def compare_document_versions(
    previous_text: str,
    current_text: str
) -> Dict[str, Any]:

    all_fields = list(set(
        APPLICATION_REQUIRED_FIELDS +
        SAE_REQUIRED_FIELDS +
        CONSISTENCY_FIELDS
    ))

    text_diff = compare_text_versions(previous_text, current_text)
    field_diff = compare_field_changes(previous_text, current_text, all_fields)
    table_diff = compare_table_like_content(previous_text, current_text)

    significant_flags = flag_significant_changes(
        text_diff=text_diff,
        field_diff=field_diff,
        table_diff=table_diff
    )

    return {
        "text_diff": text_diff,
        "field_diff": field_diff,
        "table_diff": table_diff,
        "significant_flags": significant_flags
    }


# ============================================================
# RUN COMPLETE FOLDER PIPELINE
# ============================================================

def run_completeness_and_comparison(
    current_folder: str,
    previous_folder: str = None,
    output_folder: str = OUTPUT_FOLDER,
    document_type: str = "APPLICATION"
):

    Path(output_folder).mkdir(parents=True, exist_ok=True)

    current_docs = load_folder_texts(current_folder)

    final_report = {
        "document_type": document_type,
        "total_current_documents": len(current_docs),
        "document_reports": [],
        "consistency_report": {},
        "version_comparison_reports": []
    }

    # 1. Completeness for each current document
    for file_name, text in current_docs.items():
        doc_report = assess_single_document(
            file_name=file_name,
            text=text,
            document_type=document_type
        )

        final_report["document_reports"].append(doc_report)

    # 2. Consistency across current documents
    final_report["consistency_report"] = check_consistency_across_documents(
        documents=current_docs,
        consistency_fields=CONSISTENCY_FIELDS
    )

    # 3. Version comparison if previous folder exists
    if previous_folder and Path(previous_folder).exists():
        previous_docs = load_folder_texts(previous_folder)

        for file_name, current_text in current_docs.items():
            if file_name in previous_docs:
                comparison = compare_document_versions(
                    previous_text=previous_docs[file_name],
                    current_text=current_text
                )

                final_report["version_comparison_reports"].append({
                    "file_name": file_name,
                    "comparison": comparison
                })

    output_path = Path(output_folder) / "completeness_comparison_report.json"

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(final_report, f, indent=2, ensure_ascii=False)

    print_summary(final_report)
    print(f"\nSaved report to: {output_path}")

    return final_report


# ============================================================
# PRINT SUMMARY
# ============================================================

def print_summary(report: Dict[str, Any]):

    print("\n📋 COMPLETENESS + DOCUMENT COMPARISON REPORT")
    print("=" * 70)

    print(f"Document Type: {report['document_type']}")
    print(f"Total Current Documents: {report['total_current_documents']}")

    print("\n📌 Completeness")
    for doc in report["document_reports"]:
        c = doc["completeness"]
        print(f"\nFile: {doc['file_name']}")
        print(f"Completeness Score: {c['completeness_score']}")
        print(f"Status: {c['status']}")
        print(f"Missing Fields: {c['missing_fields']}")

    print("\n🔁 Consistency")
    consistency = report["consistency_report"]
    print(f"Status: {consistency['status']}")

    if consistency["inconsistencies"]:
        for issue in consistency["inconsistencies"]:
            print(f"- Field mismatch: {issue['field']}")
            print(f"  Values: {issue['values_by_file']}")

    print("\n🧾 Version Comparison")
    if not report["version_comparison_reports"]:
        print("No previous matching versions found.")
    else:
        for item in report["version_comparison_reports"]:
            comp = item["comparison"]
            print(f"\nFile: {item['file_name']}")
            print(f"Text Similarity: {comp['text_diff']['similarity_score']}")
            print(f"Added Lines: {comp['text_diff']['total_added_lines']}")
            print(f"Removed Lines: {comp['text_diff']['total_removed_lines']}")
            print(f"Field Changes: {comp['field_diff']['total_field_changes']}")
            print(f"Table Rows Added: {comp['table_diff']['total_added_table_rows']}")
            print(f"Table Rows Removed: {comp['table_diff']['total_removed_table_rows']}")
            print(f"Significant Change Status: {comp['significant_flags']['status']}")

            for flag in comp["significant_flags"]["significant_change_flags"]:
                print(f"- {flag['message']}")

    print("=" * 70)


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    # For clinical approval application:
    run_completeness_and_comparison(
        current_folder=CURRENT_FOLDER,
        previous_folder=PREVIOUS_FOLDER,
        output_folder=OUTPUT_FOLDER,
        document_type="APPLICATION"
    )

    # For SAE reports, use:
    # run_completeness_and_comparison(
    #     current_folder=CURRENT_FOLDER,
    #     previous_folder=PREVIOUS_FOLDER,
    #     output_folder=OUTPUT_FOLDER,
    #     document_type="SAE"
    # )

In [5]:
# case_classification_tool.py

import re
import json
from pathlib import Path
from difflib import SequenceMatcher
from typing import Dict, Any, List


# ============================================================
# CONFIG
# ============================================================

INPUT_FOLDER = "/content/final_outputs/anonymized"
OUTPUT_FOLDER = "/content/case_classification_outputs"

DUPLICATE_SIMILARITY_THRESHOLD = 0.85


# ============================================================
# SEVERITY KEYWORDS
# ============================================================

SEVERITY_RULES = {
    "DEATH": [
        "death", "died", "fatal", "expired", "mortality"
    ],
    "DISABILITY": [
        "disability", "disabled", "permanent impairment",
        "persistent impairment", "congenital anomaly"
    ],
    "HOSPITALISATION": [
        "hospitalisation", "hospitalization", "admitted",
        "inpatient", "icu", "intensive care", "prolonged hospitalization"
    ],
    "LIFE_THREATENING": [
        "life threatening", "life-threatening", "critical condition",
        "ventilator", "resuscitation"
    ],
    "OTHER_SERIOUS": [
        "serious adverse event", "sae", "medically significant",
        "intervention required"
    ]
}


PRIORITY_SCORE = {
    "DEATH": 100,
    "LIFE_THREATENING": 90,
    "DISABILITY": 80,
    "HOSPITALISATION": 70,
    "OTHER_SERIOUS": 60,
    "OTHERS": 30
}


# ============================================================
# LOAD TEXT FILES
# ============================================================

def load_case_files(folder: str) -> List[Dict[str, Any]]:
    cases = []

    for file in Path(folder).glob("*.txt"):
        text = file.read_text(encoding="utf-8", errors="ignore")

        cases.append({
            "file_name": file.name,
            "text": text
        })

    return cases


# ============================================================
# FIELD EXTRACTION
# ============================================================

def extract_field(text: str, field_name: str) -> str:
    pattern = rf"{re.escape(field_name)}\s*[:\-]\s*(.+)"
    match = re.search(pattern, text, flags=re.IGNORECASE)

    if match:
        return match.group(1).strip()

    return ""


def extract_case_metadata(text: str) -> Dict[str, str]:
    return {
        "patient_id": extract_field(text, "Patient ID"),
        "subject_id": extract_field(text, "Subject ID"),
        "sae_report_number": extract_field(text, "SAE Report Number"),
        "event_term": extract_field(text, "Event Term"),
        "date_of_onset": extract_field(text, "Date of Onset"),
        "date_of_report": extract_field(text, "Date of Report"),
        "suspected_drug": extract_field(text, "Suspected Drug"),
        "outcome": extract_field(text, "Outcome"),
        "causality": extract_field(text, "Causality Assessment"),
        "site": extract_field(text, "Trial Site"),
    }


# ============================================================
# SEVERITY CLASSIFICATION
# ============================================================

def classify_severity(text: str) -> Dict[str, Any]:
    text_lower = text.lower()

    matched_categories = []

    for severity, keywords in SEVERITY_RULES.items():
        matched_keywords = [
            kw for kw in keywords if kw.lower() in text_lower
        ]

        if matched_keywords:
            matched_categories.append({
                "severity": severity,
                "matched_keywords": matched_keywords
            })

    if not matched_categories:
        final_severity = "OTHERS"
        score = PRIORITY_SCORE["OTHERS"]
    else:
        ranked = sorted(
            matched_categories,
            key=lambda x: PRIORITY_SCORE.get(x["severity"], 0),
            reverse=True
        )
        final_severity = ranked[0]["severity"]
        score = PRIORITY_SCORE[final_severity]

    return {
        "severity": final_severity,
        "priority_score": score,
        "matched_categories": matched_categories
    }


# ============================================================
# DUPLICATE DETECTION
# ============================================================

def normalize_for_duplicate(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\[[A-Z_]+(?:_\d+)?(?:_[a-f0-9]{8})?\]", "TOKEN", text)
    text = re.sub(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", "DATE", text)
    text = re.sub(r"\d{4}-\d{2}-\d{2}", "DATE", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def metadata_duplicate_score(meta1: Dict[str, str], meta2: Dict[str, str]) -> float:
    important_fields = [
        "patient_id",
        "subject_id",
        "sae_report_number",
        "event_term",
        "date_of_onset",
        "suspected_drug"
    ]

    matches = 0
    available = 0

    for field in important_fields:
        v1 = meta1.get(field, "").lower().strip()
        v2 = meta2.get(field, "").lower().strip()

        if v1 or v2:
            available += 1
            if v1 and v2 and v1 == v2:
                matches += 1

    if available == 0:
        return 0.0

    return matches / available


def text_similarity(text1: str, text2: str) -> float:
    t1 = normalize_for_duplicate(text1)
    t2 = normalize_for_duplicate(text2)

    return SequenceMatcher(None, t1, t2).ratio()


def detect_duplicates(cases: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    duplicate_pairs = []

    for i in range(len(cases)):
        for j in range(i + 1, len(cases)):
            case1 = cases[i]
            case2 = cases[j]

            meta_score = metadata_duplicate_score(
                case1["metadata"],
                case2["metadata"]
            )

            sim_score = text_similarity(
                case1["text"],
                case2["text"]
            )

            final_score = max(meta_score, sim_score)

            if final_score >= DUPLICATE_SIMILARITY_THRESHOLD:
                duplicate_pairs.append({
                    "case_1": case1["file_name"],
                    "case_2": case2["file_name"],
                    "metadata_score": round(meta_score, 4),
                    "text_similarity_score": round(sim_score, 4),
                    "duplicate_score": round(final_score, 4),
                    "status": "LIKELY_DUPLICATE"
                })

    return duplicate_pairs


# ============================================================
# PRIORITISATION
# ============================================================

def assign_review_priority(case: Dict[str, Any]) -> Dict[str, Any]:
    score = case["severity_result"]["priority_score"]
    metadata = case["metadata"]

    if metadata.get("outcome", "").lower() in ["fatal", "death", "died"]:
        score += 20

    if metadata.get("causality", "").lower() in [
        "related", "probable", "possible", "suspected"
    ]:
        score += 10

    if not metadata.get("date_of_onset"):
        score += 5

    if not metadata.get("causality"):
        score += 5

    if score >= 100:
        priority = "URGENT"
    elif score >= 80:
        priority = "HIGH"
    elif score >= 60:
        priority = "MEDIUM"
    else:
        priority = "LOW"

    return {
        "priority": priority,
        "priority_score": score
    }


# ============================================================
# MAIN PIPELINE
# ============================================================

def classify_cases(input_folder: str, output_folder: str) -> Dict[str, Any]:
    Path(output_folder).mkdir(parents=True, exist_ok=True)

    cases = load_case_files(input_folder)

    enriched_cases = []

    for case in cases:
        metadata = extract_case_metadata(case["text"])
        severity_result = classify_severity(case["text"])

        enriched = {
            "file_name": case["file_name"],
            "metadata": metadata,
            "severity_result": severity_result,
            "text": case["text"]
        }

        priority_result = assign_review_priority(enriched)

        enriched["priority_result"] = priority_result

        enriched_cases.append(enriched)

    duplicate_pairs = detect_duplicates(enriched_cases)

    duplicate_files = set()
    for pair in duplicate_pairs:
        duplicate_files.add(pair["case_2"])

    review_queue = []

    for case in enriched_cases:
        review_queue.append({
            "file_name": case["file_name"],
            "severity": case["severity_result"]["severity"],
            "priority": case["priority_result"]["priority"],
            "priority_score": case["priority_result"]["priority_score"],
            "is_possible_duplicate": case["file_name"] in duplicate_files,
            "metadata": case["metadata"]
        })

    review_queue = sorted(
        review_queue,
        key=lambda x: x["priority_score"],
        reverse=True
    )

    report = {
        "total_cases": len(enriched_cases),
        "duplicate_pairs": duplicate_pairs,
        "review_queue": review_queue,
        "case_details": enriched_cases
    }

    output_path = Path(output_folder) / "case_classification_report.json"

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    print_case_summary(report)
    print(f"\nSaved report to: {output_path}")

    return report


# ============================================================
# PRINT SUMMARY
# ============================================================

def print_case_summary(report: Dict[str, Any]):
    print("\n🚦 CASE CLASSIFICATION + PRIORITISATION REPORT")
    print("=" * 70)

    print(f"Total Cases: {report['total_cases']}")
    print(f"Likely Duplicate Pairs: {len(report['duplicate_pairs'])}")

    print("\n📌 Review Queue")
    for item in report["review_queue"]:
        print(
            f"{item['priority']:8} | "
            f"Score: {item['priority_score']:3} | "
            f"Severity: {item['severity']:18} | "
            f"Duplicate: {item['is_possible_duplicate']} | "
            f"File: {item['file_name']}"
        )

    if report["duplicate_pairs"]:
        print("\n🔁 Likely Duplicates")
        for pair in report["duplicate_pairs"]:
            print(
                f"{pair['case_1']} ↔ {pair['case_2']} "
                f"| Score: {pair['duplicate_score']}"
            )

    print("=" * 70)


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    classify_cases(
        input_folder=INPUT_FOLDER,
        output_folder=OUTPUT_FOLDER
    )


🚦 CASE CLASSIFICATION + PRIORITISATION REPORT
Total Cases: 1
Likely Duplicate Pairs: 0

📌 Review Queue
LOW      | Score:  40 | Severity: OTHERS             | Duplicate: False | File: good_application_1_anonymized.txt

Saved report to: /content/case_classification_outputs/case_classification_report.json


In [ ]:
# inspection_report_generation.py

import re
import json
from pathlib import Path
from datetime import datetime


INPUT_FOLDER = "/content/regulatory_summaries"
OUTPUT_FOLDER = "/content/inspection_reports"


GCP_TEMPLATE = {
    "I. General": [
        "Name and address of the clinical trial site",
        "Date of Inspection",
        "Inspection Team Members",
        "Personnel present during Inspection",
        "Address & Contact details of Investigator",
        "Name & address of the Sponsor",
        "Name & address of clinical trial NOC holder",
        "Name & address of EC",
        "Protocol Title",
        "Protocol Number / Version / Date / Amendments",
        "Investigational Product",
        "Stage of study",
        "Type of Inspection"
    ],

    "II. Legal & Administrative Aspects": [
        "Clinical trial NOC from O/o DCGI",
        "NOC for subsequent protocol amendments",
        "Ethics Committee approval date",
        "Appendix VII as per Schedule Y",
        "Valid financial agreement between Sponsor, Investigator and Institution",
        "Liability of involved parties clearly agreed",
        "Valid clinical trial insurance available",
        "Site initiation date",
        "Date of screening of first subject",
        "Date of signing ICF by first subject",
        "Date of Last Patient Last Follow-Up",
        "SOPs for various activities established and documented",
        "Adequate emergency care facilities"
    ],

    "III. Organisation & Personnel": [
        "Signed and dated CV available for Investigator/Sub-Investigator",
        "Investigator qualification and Medical Council registration verified",
        "GCP, Schedule Y and protocol-specific training verified",
        "Duty delegation log available",
        "Delegated personnel qualified and trained",
        "List of clinical trials performed by Investigator",
        "Investigator involved in not more than three trials"
    ],

    "IV. Conduct of Trial": [
        "Informed consent for screening reviewed",
        "Screening log and enrolment log checked",
        "Inclusion/exclusion criteria verified",
        "Clinical examination verified",
        "Laboratory evaluation verified",
        "X-Ray/MRI/ECG/USG verification",
        "Clinical trial NOC conditions followed",
        "ICF elements as per Schedule Y verified",
        "ICF approved by Ethics Committee",
        "Consent obtained before participation",
        "Subject/legal representative signature with date",
        "Witness signature where applicable",
        "Investigator signature on completed ICF",
        "Re-consenting performed if ICF changed",
        "Source documents complete and legible",
        "Source document compared with CRF",
        "SAE reporting SOP available",
        "SAEs reported to Sponsor, EC and Licensing Authority",
        "Medical care during SAE verified"
    ],

    "VI. Sponsor": [
        "Investigator maintains copies of reports submitted to sponsor",
        "CRFs submitted to sponsor",
        "Dropouts reported to sponsor",
        "Monitoring method and frequency verified",
        "Qualified monitor appointed",
        "Onsite monitoring visit log maintained",
        "Monitor visit report available",
        "Sponsor audit performed",
        "Premature termination/suspension notified"
    ],

    "VII. Investigational Product": [
        "Correct dose, frequency and route verified",
        "Unauthorized dispensing checked",
        "Drug accountability/reconciliation records maintained",
        "Storage conditions monitored",
        "Trial medication secured with controlled access",
        "Unused trial medication returned/disposed",
        "Dispensing records maintained",
        "IP reconciliation records maintained",
        "Temperature logs available",
        "Investigational product labelled appropriately"
    ],

    "VIII. Ethics Committee": [
        "EC name/address verified",
        "EC registration status verified",
        "EC approval letter complete",
        "EC meeting minutes recorded",
        "EC onsite monitoring verified",
        "Conflict of interest checked",
        "Communication between Investigator and EC available",
        "EC functioning as per registration conditions"
    ],

    "IX. Pathology Laboratory": [
        "Name/address of laboratory recorded",
        "Financial/confidentiality agreement available",
        "Accreditation and facility adequacy verified",
        "SOP for sample preparation, handling and transportation available"
    ],

    "X. Quality Assurance": [
        "SOPs for all site procedures available",
        "SOP preparation/check/authorization/revision verified",
        "SOPs for screening, ICF, AV consent, SAE, EC/Sponsor/CDSCO communication available",
        "SOPs for IP handling and sample processing available",
        "SOPs for storage cabinets/refrigerators/freezers available",
        "Training and responsibility records maintained",
        "Activities comply with duty delegation",
        "Staff adequately trained",
        "Vaccine spillage SOP available if applicable"
    ],

    "XI. Record Keeping and Data Handling": [
        "Adequate document retention space available",
        "Documents maintained for specified period",
        "Measures against accidental destruction available",
        "Archival access controlled",
        "Data management SOP available",
        "Corrections carry date and initials"
    ],

    "XI-a Electronic Data Processing": [
        "Electronic data processing done by authorized person",
        "List of authorized persons maintained",
        "Audit trail for changes/deletions available",
        "Hardware and software validated"
    ]
}


def read_text_files(folder):
    docs = []
    for file in Path(folder).glob("*.txt"):
        docs.append({
            "file_name": file.name,
            "text": file.read_text(encoding="utf-8", errors="ignore")
        })
    return docs


def detect_status(observation):
    obs = observation.lower()

    negative_words = [
        "not available", "missing", "not maintained", "not verified",
        "not done", "not documented", "absent", "incomplete",
        "no evidence", "deviation", "non-compliance", "non compliance"
    ]

    positive_words = [
        "available", "maintained", "verified", "documented",
        "done", "present", "complied", "adequate"
    ]

    if any(w in obs for w in negative_words):
        return "No"

    if any(w in obs for w in positive_words):
        return "Yes"

    if "not applicable" in obs or "na" == obs.strip():
        return "NA"

    return "Review"


def classify_severity(observation):
    obs = observation.lower()

    critical = [
        "death", "fatal", "serious adverse event not reported",
        "consent not obtained", "fraud", "fabricated",
        "unauthorized", "no ethics committee approval",
        "no dcgi approval"
    ]

    major = [
        "missing", "incomplete", "not maintained", "not documented",
        "not available", "delay", "deviation", "incorrect",
        "mismatch", "not signed", "not dated"
    ]

    if any(k in obs for k in critical):
        return "Critical"

    if any(k in obs for k in major):
        return "Major"

    return "Minor/Observation"


def find_relevant_observation(item, full_text):
    item_keywords = [
        word.lower()
        for word in re.findall(r"[A-Za-z]{4,}", item)
    ]

    sentences = re.split(r"(?<=[.!?])\s+|\n+", full_text)

    best_sentence = ""
    best_score = 0

    for sentence in sentences:
        s = sentence.lower()
        score = sum(1 for kw in item_keywords if kw in s)

        if score > best_score:
            best_score = score
            best_sentence = sentence.strip()

    if best_score == 0:
        return ""

    return best_sentence


def generate_structured_report(observation_text):
    report = {
        "report_title": "GCP Inspection Report",
        "generated_on": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "sections": []
    }

    for section_name, checklist_items in GCP_TEMPLATE.items():
        section = {
            "section": section_name,
            "items": []
        }

        for idx, item in enumerate(checklist_items, start=1):
            observation = find_relevant_observation(item, observation_text)

            status = detect_status(observation) if observation else "Review"
            severity = classify_severity(observation) if observation else "Review"

            section["items"].append({
                "s_no": idx,
                "checklist_item": item,
                "status": status,
                "severity": severity,
                "observation": observation if observation else "No clear observation found. Reviewer validation required.",
                "recommendation": make_recommendation(status, severity, item)
            })

        report["sections"].append(section)

    return report


def make_recommendation(status, severity, item):
    if status == "Yes":
        return "Compliant. No immediate action required."

    if status == "NA":
        return "Not applicable. Confirm justification."

    if severity == "Critical":
        return f"Immediate corrective action required for: {item}"

    if severity == "Major":
        return f"Corrective and preventive action required for: {item}"

    return f"Reviewer should verify and document evidence for: {item}"


def report_to_markdown(report):
    lines = []

    lines.append("# GCP Inspection Report")
    lines.append("")
    lines.append(f"Generated On: {report['generated_on']}")
    lines.append("")
    lines.append("## Summary")
    lines.append("")

    total = 0
    yes = no = review = critical = major = 0

    for section in report["sections"]:
        for item in section["items"]:
            total += 1
            if item["status"] == "Yes":
                yes += 1
            elif item["status"] == "No":
                no += 1
            elif item["status"] == "Review":
                review += 1

            if item["severity"] == "Critical":
                critical += 1
            elif item["severity"] == "Major":
                major += 1

    lines.append(f"- Total checklist items: {total}")
    lines.append(f"- Compliant items: {yes}")
    lines.append(f"- Non-compliant items: {no}")
    lines.append(f"- Items requiring review: {review}")
    lines.append(f"- Critical findings: {critical}")
    lines.append(f"- Major findings: {major}")
    lines.append("")

    for section in report["sections"]:
        lines.append(f"## {section['section']}")
        lines.append("")
        lines.append("| S.No | Checklist Item | Yes/No/NA | Severity | Observation | Recommendation |")
        lines.append("|---|---|---|---|---|---|")

        for item in section["items"]:
            lines.append(
                f"| {item['s_no']} "
                f"| {clean_md(item['checklist_item'])} "
                f"| {item['status']} "
                f"| {item['severity']} "
                f"| {clean_md(item['observation'])} "
                f"| {clean_md(item['recommendation'])} |"
            )

        lines.append("")

    return "\n".join(lines)


def clean_md(text):
    return str(text).replace("|", "/").replace("\n", " ").strip()


def process_inspection_folder(input_folder, output_folder):
    Path(output_folder).mkdir(parents=True, exist_ok=True)

    docs = read_text_files(input_folder)

    for doc in docs:
        base = Path(doc["file_name"]).stem

        report = generate_structured_report(doc["text"])
        markdown = report_to_markdown(report)

        json_path = Path(output_folder) / f"{base}_inspection_report.json"
        md_path = Path(output_folder) / f"{base}_inspection_report.md"

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2, ensure_ascii=False)

        with open(md_path, "w", encoding="utf-8") as f:
            f.write(markdown)

        print(f"Generated:")
        print(f"- {json_path}")
        print(f"- {md_path}")

if __name__ == "__main__":
    process_inspection_folder(
        input_folder=INPUT_FOLDER,
        output_folder=OUTPUT_FOLDER
    )